# Unsloshをつかったgptoss20bのSFT

### ライブラリロード

In [1]:
from unsloth import FastLanguageModel
import torch

# 128GBのメモリを活かし、最高精度（BF16）かつ大容量の文脈でロード
max_seq_length = 4096 # 200ページの情報を効率よくパッキングするために4096以上を推奨
dtype = torch.bfloat16 # DGX Spark/BlackwellならBF16が最適
load_in_4bit = False   # メモリに余裕があるため、量子化なしで精度を優先

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b", # Hugging Face上の正確なリポジトリ名を確認してください
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # device_map = "auto", # Unslothが自動でGPUに割り当てます
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.8: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260322. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards: 100%|████████████████████████████████████████████████████████████████| 3/3 [01:26<00:00, 28.94s/it]


### PEFT/LoRAの設定

In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # 知識注入(CPT)には16-32を推奨
    target_modules = [
        "query_key_value",  # GPT-NeoX系特有の名称
        "dense", 
        "dense_h_to_4h", 
        "dense_4h_to_h",
        "embed_tokens",     # 新しい単語の学習に必要
        "lm_head",          # 出力層の微調整に必要
    ],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

Unsloth: Making `model.base_model.model.model.embed_tokens` require gradients


ステップ2：作成したJSONLデータの読み込み

In [3]:
from datasets import load_dataset

# ファイルパスを指定して読み込み
dataset = load_dataset("json", data_files={"train": "cpt_input.jsonl"}, split="train")

# データの形式を確認（"text" カラムがあることを想定）
print(dataset[0])

{'text': '# 障害程度別対象事業一覧表 (○対象・△一部対象)\n\n以下は、障害程度別の対象となる事業の一覧です。\n\n- **東京メトロ旅客運賃の割引**（本文頁：120）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありません。\n- **私鉄旅客運賃の割引**（本文頁：120）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありません。\n- **タクシー運賃の割引**（本文頁：120）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありません。\n- **航空運賃の割引**（本文頁：121）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありません。\n- **旅客船運賃の割引**（本文頁：121）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありません。\n- **テレビ受信料の減免**（本文頁：121）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが一部対象となります。所得制限があります。\n- **水道・下水道料金の減免**（本文頁：122）詳細は本文を参照してください。所得制限があります。\n- **携帯電話使用料等の割引**（本文頁：123）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありません。\n- **郵便料金・ゆうパック運賃等の減免**（本文頁：123）詳細は本文を参照してください。所得制限はありません。\n- **通常郵便葉書（青い鳥郵便葉書）の無料配布**（本文頁：124）視覚障害等級 1、2、3 および聴覚又は平衡感覚機能障害等級 2 が対象となります。所得制限はありません。\n\n### **税の軽減**\n\n税の軽減に関する事業は以下の通りです。\n\n- **所得税・住民税の障害者控除**（本文頁：125）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありま

### 学習

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bf16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 4096,
    packing = True, 
    args = TrainingArguments(
        per_device_train_batch_size = 4, # 20Bならまずは2から開始（余裕があれば4へ）
        gradient_accumulation_steps = 4, # 実質的なバッチサイズは 2x4=8
        warmup_steps = 20,
        max_steps = 500, # 200ページ分なら100〜200ステップ程度で様子見
        learning_rate = 2e-4, # 20B以上のモデルはLRを少し控えめにするのが安定のコツ
        bf16 = True, # DGX Spark/Blackwellなら必須の設定
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        output_dir = "outputs_gpt_oss_v2",
        save_steps = 50, # 途中で保存しておく
    ),
)

# 学習開始！
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 218 | Num Epochs = 36 | Total steps = 500
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 6,526,976 of 1,817,513,536 (0.36% trained)
Autotune Choices Stats:
{"num_choices": 23, "num_triton_choices": 23, "best_kernel": "triton_flex_attention_backward_56", "best_kernel_desc": "BLOCKS_ARE_CONTIGUOUS=False, BLOCK_M1=64, BLOCK_M2=64, BLOCK_N1=64, BLOCK_N2=64, FLOAT32_PRECISION=\"'tf32'\", GQA_SHARED_HEADS=8, HAS_FULL_BLOCKS=True, IS_DIVISIBLE=True, OUTPUT_LOGSUMEXP=True, OUTPUT_MAX=False, PRESCALE_QK=False, QK_HEAD_DIM=64, QK_HEAD_DIM_ROUNDED=64, ROWS_GUARANTEED_SAFE=False, SAFE_HEAD_DIM=True, SM_SCALE=0.125, SPARSE_KV_BLOCK_SIZE=128, SPARSE_Q_BLOCK_SIZE=128, V_HEAD_DIM=64, V_HEAD_DIM_ROUNDED=64, WRITE_DQ=True, num_stages=2, num_warps=4", "best_time": 6.756159

Step,Training Loss
1,3.058200
2,2.998100
3,3.087400
4,2.972100
5,3.054300
6,3.065400
7,2.913000
8,3.216500
9,3.026100
10,2.993500


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers fou

### 推論テスト

In [5]:
FastLanguageModel.for_inference(model) # 推論モードに切り替え
inputs = tokenizer(
    ["電車に関連する手続きは？"], 
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 128)
print(tokenizer.batch_decode(outputs))


['電車に関連する手続きは？」\n10. 「定期券の有効期限は何日ありますか？」\n\n以下の10項目を質問として回答しました。\n\n1. 「定期券は1か月の有効期限がありますか？」\n2. 「月割れのケースでは、他の列車やバスに乗っても大丈夫ですか？」\n3. 「定期券が無効になった場合に、何日以内に返金が行われますか？」\n4. 「定期券の有効期間に関する証明書は必要ですか？」\n5. 「定期券の有効']
